In [11]:
# Load TTC Bronze datasets

df_subway = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/bronze/subway/ttc_subway_delays_since_2025.csv")
)

df_bus = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/bronze/bus/ttc_bus_delays_since_2025.csv")
)

df_streetcar = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/bronze/streetcar/ttc_streetcar_delays_since_2025.csv")
)

df_lrt = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/bronze/lrt/ttc_lrt_delays.csv")
)

print("Subway rows:", df_subway.count())
print("Bus rows:", df_bus.count())
print("Streetcar rows:", df_streetcar.count())
print("LRT rows:", df_lrt.count())

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 13, Finished, Available, Finished, False)

Subway rows: 45475
Bus rows: 102257
Streetcar rows: 23703
LRT rows: 4569


In [12]:
from pyspark.sql import functions as F

def standardize_ttc(df, mode):

    return (
        df
        .withColumnRenamed("_id", "IncidentID")
        .withColumnRenamed("Date", "DelayDate")
        .withColumnRenamed("Time", "DelayTime")
        .withColumnRenamed("Day", "DayOfWeek")
        .withColumnRenamed("Station", "Location")
        .withColumnRenamed("Code", "IncidentCode")
        .withColumnRenamed("Min Delay", "DelayMinutes")
        .withColumnRenamed("Min Gap", "GapMinutes")
        .withColumnRenamed("Bound", "Direction")
        .withColumnRenamed("Line", "RouteLine")
        .withColumnRenamed("Vehicle", "VehicleID")
        .withColumn("Mode", F.lit(mode))
    )


silver_subway = standardize_ttc(df_subway, "Subway")
silver_bus = standardize_ttc(df_bus, "Bus")
silver_streetcar = standardize_ttc(df_streetcar, "Streetcar")
silver_lrt = standardize_ttc(df_lrt, "LRT")
silver_subway.printSchema()

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 14, Finished, Available, Finished, False)

root
 |-- IncidentID: integer (nullable = true)
 |-- DelayDate: date (nullable = true)
 |-- DelayTime: timestamp (nullable = true)
 |-- DayOfWeek: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- IncidentCode: string (nullable = true)
 |-- DelayMinutes: integer (nullable = true)
 |-- GapMinutes: integer (nullable = true)
 |-- Direction: string (nullable = true)
 |-- RouteLine: string (nullable = true)
 |-- VehicleID: integer (nullable = true)
 |-- Mode: string (nullable = false)



In [13]:
def clean_ttc_silver(df):

    return (
        df

        # Trim whitespace from text columns
        .withColumn("DayOfWeek", F.trim(F.col("DayOfWeek")))
        .withColumn("Location", F.trim(F.col("Location")))
        .withColumn("IncidentCode", F.trim(F.col("IncidentCode")))
        .withColumn("Direction", F.trim(F.col("Direction")))
        .withColumn("RouteLine", F.trim(F.col("RouteLine")))

        # Replace missing categorical values
        .fillna({
            "Location": "Unknown",
            "IncidentCode": "Unknown",
            "Direction": "Unknown",
            "RouteLine": "Unknown"
        })
    )


silver_subway = clean_ttc_silver(silver_subway)
silver_bus = clean_ttc_silver(silver_bus)
silver_streetcar = clean_ttc_silver(silver_streetcar)
silver_lrt = clean_ttc_silver(silver_lrt)

print("Silver text cleaning completed.")

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 15, Finished, Available, Finished, False)

Silver text cleaning completed.


In [14]:
def add_time_features(df):

    return (
        df

        # Calendar attributes
        .withColumn("Year", F.year("DelayDate"))
        .withColumn("MonthNumber", F.month("DelayDate"))
        .withColumn("MonthName", F.date_format("DelayDate", "MMMM"))
        .withColumn("YearMonth", F.date_format("DelayDate", "yyyy-MM"))
        .withColumn("Hour", F.hour("DelayTime"))

        # Time-of-day grouping
        .withColumn(
            "TimePeriod",
            F.when(F.col("Hour").between(5, 11), "Morning")
             .when(F.col("Hour").between(12, 16), "Afternoon")
             .when(F.col("Hour").between(17, 20), "Evening")
             .otherwise("Night")
        )

        # Flag incidents that actually recorded a positive delay
        .withColumn(
            "HasDelay",
            F.when(F.col("DelayMinutes") > 0, 1).otherwise(0)
        )
    )


silver_subway = add_time_features(silver_subway)
silver_bus = add_time_features(silver_bus)
silver_streetcar = add_time_features(silver_streetcar)
silver_lrt = add_time_features(silver_lrt)

print("Time features created.")

display(
    silver_subway.select(
        "DelayDate",
        "DelayTime",
        "DayOfWeek",
        "Year",
        "MonthNumber",
        "MonthName",
        "YearMonth",
        "Hour",
        "TimePeriod",
        "DelayMinutes",
        "HasDelay"
    ).limit(10)
)

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 16, Finished, Available, Finished, False)

Time features created.


SynapseWidget(Synapse.DataFrame, 1662a66e-5e5f-4f44-bd5c-22709ae6e071)

In [15]:
def fix_delay_time(df):

    return (
        df
        .withColumn(
            "DelayTime",
            F.date_format(F.col("DelayTime"), "HH:mm:ss")
        )
    )


silver_subway = fix_delay_time(silver_subway)
silver_bus = fix_delay_time(silver_bus)
silver_streetcar = fix_delay_time(silver_streetcar)
silver_lrt = fix_delay_time(silver_lrt)

print("DelayTime standardized.")

# display(
#     silver_subway.select(
#         "DelayDate",
#         "DelayTime",
#         "Hour",
#         "TimePeriod"
#     ).limit(10)
# )

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 17, Finished, Available, Finished, False)

DelayTime standardized.


In [16]:
import requests
import time

# Subway Toronto Open Data package
subway_package_id = "996cfe8d-fb35-40ce-b569-698d51fc683b"

subway_api_url = (
    "https://ckan0.cf.opendata.inter.prod-toronto.ca/"
    f"api/3/action/package_show?id={subway_package_id}"
)

package_retrieved = False

for attempt in range(3):
    try:
        response = requests.get(subway_api_url, timeout=120)
        response.raise_for_status()

        subway_package = response.json()["result"]

        print("Subway package retrieved successfully")

        package_retrieved = True
        break

    except requests.exceptions.RequestException as e:
        print(f"Attempt {attempt + 1} failed:", e)

        if attempt < 2:
            print("Retrying in 10 seconds...")
            time.sleep(10)

if not package_retrieved:
    raise Exception(
        "Subway code-description package retrieval failed after 3 attempts. "
        "Toronto Open Data API is currently unavailable."
    )

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 18, Finished, Available, Finished, False)

Subway package retrieved successfully


In [17]:
import unicodedata
import pandas as pd

def clean_description(value):
    if pd.isna(value):
        return value

    value = str(value)

    try:
        value = value.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass

    value = unicodedata.normalize("NFKC", value)

    value = "".join(
        char for char in value
        if char.isprintable()
    )

    value = " ".join(value.split())

    return value

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 19, Finished, Available, Finished, False)

In [18]:
# Select Subway Code Descriptions resource

subway_code_resource = next(
    resource
    for resource in subway_package["resources"]
    if resource.get("name") == "Code Descriptions.csv"
    and resource.get("format", "").upper() == "CSV"
)

subway_code_url = subway_code_resource["url"]

# Load lookup CSV into pandas
import pandas as pd

subway_codes_pd = pd.read_csv(subway_code_url)

# Clean corrupted/non-printable characters in descriptions
subway_codes_pd["DESCRIPTION"] = (
    subway_codes_pd["DESCRIPTION"]
    .apply(clean_description)
)

# Convert cleaned lookup to Spark
df_subway_codes = spark.createDataFrame(subway_codes_pd)

print("Subway code descriptions loaded and cleaned successfully.")

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 20, Finished, Available, Finished, False)

Subway code descriptions loaded and cleaned successfully.


In [19]:
# Prepare Subway incident code lookup

subway_codes_clean = (
    df_subway_codes
    .select(
        F.trim(F.col("CODE")).alias("IncidentCode"),
        F.trim(F.col("DESCRIPTION")).alias("IncidentDescription")
    )
    .dropDuplicates(["IncidentCode"])
)

# Join descriptions to Subway delay data
silver_subway = (
    silver_subway
    .join(
        subway_codes_clean,
        on="IncidentCode",
        how="left"
    )
    .fillna({
        "IncidentDescription": "Unknown"
    })
)

print("Subway incident descriptions added.")

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 21, Finished, Available, Finished, False)

Subway incident descriptions added.


In [20]:
# Package IDs for remaining TTC modes

package_ids = {
    "Bus": "e271cdae-8788-4980-96ce-6a5c95bc6618",
    "Streetcar": "b68cb71b-44a7-4394-97e2-5d2f41462a5d",
    "LRT": "d4d64b0e-2945-4c92-960e-f0103ad48b02"
}

code_urls = {}

for mode, package_id in package_ids.items():

    api_url = (
        "https://ckan0.cf.opendata.inter.prod-toronto.ca/"
        f"api/3/action/package_show?id={package_id}"
    )

    package_retrieved = False

    for attempt in range(3):
        try:
            response = requests.get(api_url, timeout=120)
            response.raise_for_status()

            package_data = response.json()["result"]

            resource = next(
                r for r in package_data["resources"]
                if r.get("format", "").upper() == "CSV"
                and "Code Descriptions" in r.get("name", "")
            )

            code_urls[mode] = resource["url"]

            print(f"{mode}: Code Descriptions.csv found")

            package_retrieved = True
            break

        except requests.exceptions.RequestException as e:
            print(f"{mode} attempt {attempt + 1} failed:", e)

            if attempt < 2:
                print("Retrying in 10 seconds...")
                time.sleep(10)

    if not package_retrieved:
        raise Exception(
            f"{mode} code-description package retrieval failed after 3 attempts. "
            "Toronto Open Data API is currently unavailable."
        )

print("Lookup discovery complete.")

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 22, Finished, Available, Finished, False)

Bus: Code Descriptions.csv found
Streetcar: Code Descriptions.csv found
LRT: Code Descriptions.csv found
Lookup discovery complete.


In [21]:
import pandas as pd
import requests
import time
from io import StringIO
from pyspark.sql import functions as F


def load_code_lookup(url, mode, max_attempts=5):

    for attempt in range(max_attempts):
        try:
            print(f"{mode} download attempt {attempt + 1}...")

            response = requests.get(url, timeout=180)
            response.raise_for_status()

            # Decode CSV as UTF-8
            text = response.content.decode("utf-8")

            # Read CSV into pandas
            pdf = pd.read_csv(StringIO(text))

            # Clean corrupted/non-printable characters in descriptions
            pdf["DESCRIPTION"] = (
                pdf["DESCRIPTION"]
                .apply(clean_description)
            )

            # Convert cleaned pandas DataFrame to Spark
            sdf = spark.createDataFrame(pdf)

            # Standardize lookup columns
            lookup = (
                sdf
                .select(
                    F.trim(F.col("CODE")).alias("IncidentCode"),
                    F.trim(F.col("DESCRIPTION")).alias("IncidentDescription")
                )
                .dropDuplicates(["IncidentCode"])
            )

            print(f"{mode} lookup loaded and cleaned successfully")
            return lookup

        except Exception as e:
            print(f"{mode} attempt {attempt + 1} failed: {e}")

            if attempt < max_attempts - 1:
                print("Waiting 15 seconds before retry...")
                time.sleep(15)

    raise Exception(
        f"{mode} lookup failed after {max_attempts} attempts."
    )


# Load official lookup tables
bus_codes = load_code_lookup(
    code_urls["Bus"],
    "Bus"
)

streetcar_codes = load_code_lookup(
    code_urls["Streetcar"],
    "Streetcar"
)

lrt_codes = load_code_lookup(
    code_urls["LRT"],
    "LRT"
)


# Join descriptions and distinguish codes classified as OTHER
def add_descriptions(df, codes):

    return (
        df
        .join(
            codes,
            on="IncidentCode",
            how="left"
        )
        .fillna({
            "IncidentDescription": "Unknown"
        })
        .withColumn(
            "IncidentDescription",
            F.when(
                F.upper(F.trim(F.col("IncidentDescription"))) == "OTHER",
                F.concat(
                    F.col("IncidentCode"),
                    F.lit(" - OTHER")
                )
            )
            .otherwise(
                F.trim(F.col("IncidentDescription"))
            )
        )
    )


# Apply descriptions to each transit mode
silver_bus = add_descriptions(
    silver_bus,
    bus_codes
)

silver_streetcar = add_descriptions(
    silver_streetcar,
    streetcar_codes
)

silver_lrt = add_descriptions(
    silver_lrt,
    lrt_codes
)


print("\nAll Bus, Streetcar and LRT descriptions cleaned and added.")

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 23, Finished, Available, Finished, False)

Bus download attempt 1...
Bus lookup loaded and cleaned successfully
Streetcar download attempt 1...
Streetcar lookup loaded and cleaned successfully
LRT download attempt 1...
LRT lookup loaded and cleaned successfully

All Bus, Streetcar and LRT descriptions cleaned and added.


In [22]:
# Standardize DelayDate data type across all modes

def standardize_date_type(df):
    return df.withColumn(
        "DelayDate",
        F.to_date(F.col("DelayDate"))
    )

silver_subway = standardize_date_type(silver_subway)
silver_bus = standardize_date_type(silver_bus)
silver_streetcar = standardize_date_type(silver_streetcar)
silver_lrt = standardize_date_type(silver_lrt)

print("DelayDate standardized across all modes.")

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 24, Finished, Available, Finished, False)

DelayDate standardized across all modes.


In [23]:
def add_delay_datetime(df):
    return df.withColumn(
        "DelayDateTime",
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.date_format(F.col("DelayDate"), "yyyy-MM-dd"),
                F.col("DelayTime")
            ),
            "yyyy-MM-dd HH:mm:ss"
        )
    )

silver_subway = add_delay_datetime(silver_subway)
silver_bus = add_delay_datetime(silver_bus)
silver_streetcar = add_delay_datetime(silver_streetcar)
silver_lrt = add_delay_datetime(silver_lrt)

print("DelayDateTime created for all modes.")

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 25, Finished, Available, Finished, False)

DelayDateTime created for all modes.


In [24]:
def add_ingestion_timestamp(df):
    return df.withColumn(
        "IngestionTimestamp",
        F.current_timestamp()
    )

silver_subway = add_ingestion_timestamp(silver_subway)
silver_bus = add_ingestion_timestamp(silver_bus)
silver_streetcar = add_ingestion_timestamp(silver_streetcar)
silver_lrt = add_ingestion_timestamp(silver_lrt)

print("IngestionTimestamp added to all Silver datasets.")

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 26, Finished, Available, Finished, False)

IngestionTimestamp added to all Silver datasets.


In [25]:
silver_subway.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_subway_delays")

silver_bus.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_bus_delays")

silver_streetcar.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_streetcar_delays")

silver_lrt.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_lrt_delays")

print("All Silver tables saved successfully.")

StatementMeta(, a3f2ea7b-21a5-4cf6-8da2-b59ac5318788, 27, Finished, Available, Finished, False)

All Silver tables saved successfully.
